# 3D reporter timelapse — 02_retained_frame_mask_review

**Feeds:** Fig 5h

**Position in the chain:** run the numbered notebooks in order

Ported unchanged from the original analysis: outputs are as they ran, and no code
cell was edited. Paths appear as `<analysis-root>/...`.


# 02 | Retained-Frame Mask Review

This notebook is for visual mask review only.

It uses the persistent phase-artifact exclusions to avoid bad late frames and
generates image-first mask comparisons for a curated set of positions.


## What To Look For

For each position:

- the frames are all retained frames
- the final panel is the last clean frame for that position
- compare whether each mask captures the organoid body without leaking into background

This notebook is intentionally light on tables and heavy on images.


In [ ]:
import math
import re
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tifffile as tiff
from IPython.display import display
from scipy import ndimage as ndi
from skimage import filters, measure, morphology

pd.set_option("display.max_columns", 200)
plt.rcParams["figure.dpi"] = 120


In [ ]:
# -------------------------------
# User configuration
# -------------------------------
cwd = Path.cwd().resolve()
root_candidates = [cwd] + list(cwd.parents[:3])
ROOT = None
for candidate in root_candidates:
    if (candidate / "data").exists() and (candidate / "scripts").exists():
        ROOT = candidate
        break
if ROOT is None:
    raise RuntimeError("Could not locate project root from current working directory.")

POSITION_MANIFEST_PATH = ROOT / "results/manifests/acquisition_position_manifest.tsv"
EXCLUSION_PATH = ROOT / "results/qc/02_phase_artifact_excluded_frames.tsv"
PREVIEW_DIR = ROOT / "results/previews/02_retained_frame_mask_review"

position_manifest = pd.read_csv(POSITION_MANIFEST_PATH, sep="\t")
exclusion_df = pd.read_csv(EXCLUSION_PATH, sep="\t")
INTERVAL_HOURS = float(position_manifest["interval_ms"].dropna().iloc[0]) / 3_600_000.0
TIME_DISPLAY_OFFSET_HOURS = 48.0

dataset_rel = position_manifest["dataset_dir"].iloc[0]
DATASET_DIR = ROOT / dataset_rel
POSITION_RE = re.compile(r"Pos(?P<index>\d+)$")

REVIEW_POSITIONS = [
    "Pos1",
    "Pos2",
    "Pos3",
    "Pos7",
    "Pos8",
    "Pos9",
    "Pos31",
    "Pos33",
    "Pos41",
    "Pos56",
    "Pos63",
]

MASK_METHODS = ["dark_otsu", "residual_dark", "edge_fill"]
WRITE_OUTPUTS = True

PREVIEW_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", ROOT)
print("Dataset dir:", DATASET_DIR)
print("Preview dir:", PREVIEW_DIR)


In [ ]:
# -------------------------------
# Helpers
# -------------------------------
def pos_index_from_label(label: str) -> int:
    match = POSITION_RE.fullmatch(label)
    if not match:
        raise ValueError(f"Unexpected position label: {label}")
    return int(match.group("index"))


def frame_path(position_label: str, time_index: int) -> Path:
    pos_index = pos_index_from_label(position_label)
    return (
        DATASET_DIR
        / position_label
        / f"img_channel000_position{pos_index:03d}_time{time_index:09d}_z000.tif"
    )


def load_phase_frame(position_label: str, time_index: int) -> np.ndarray:
    return tiff.imread(frame_path(position_label, time_index))


def display_time_hours_from_index(time_index: int | float) -> float:
    return float(time_index) * INTERVAL_HOURS + TIME_DISPLAY_OFFSET_HOURS


def format_display_hours_from_index(time_index: int | float, decimals: int = 1) -> str:
    return f"{display_time_hours_from_index(time_index):.{decimals}f} h"


def display_time_df(df: pd.DataFrame) -> pd.DataFrame:
    output = df.copy()
    if "last_clean_frame" in output.columns:
        output["last_clean_time_hours"] = output["last_clean_frame"].map(display_time_hours_from_index)
    if "review_frames" in output.columns:
        output["review_times_hours"] = output["review_frames"].apply(
            lambda frames: [round(display_time_hours_from_index(frame), 2) for frame in frames]
        )
    return output


def display_image(image: np.ndarray, low_q: float = 1.0, high_q: float = 99.0) -> np.ndarray:
    low, high = np.percentile(image, [low_q, high_q])
    if math.isclose(high, low):
        high = low + 1.0
    return np.clip((image - low) / (high - low), 0, 1)


def draw_mask_contour(ax, mask: np.ndarray, color: str = "deepskyblue", linewidth: float = 2.0) -> None:
    if np.any(mask):
        ax.contour(mask.astype(float), levels=[0.5], colors=[color], linewidths=linewidth)


def center_component(mask: np.ndarray, min_size: int = 200) -> np.ndarray:
    mask = morphology.remove_small_objects(mask.astype(bool), min_size=min_size)
    mask = ndi.binary_fill_holes(mask)
    labels = measure.label(mask)
    if labels.max() == 0:
        return mask.astype(bool)

    center = np.array(mask.shape) / 2.0
    best_label = None
    best_score = None
    for region in measure.regionprops(labels):
        area = float(region.area)
        centroid = np.array(region.centroid)
        distance = float(np.linalg.norm(centroid - center))
        score = distance - 0.002 * area
        if best_score is None or score < best_score:
            best_score = score
            best_label = region.label

    out = labels == best_label
    out = morphology.binary_closing(out, morphology.disk(5))
    out = ndi.binary_fill_holes(out)
    return out.astype(bool)


def phase_mask_candidates(phase_image: np.ndarray) -> dict[str, np.ndarray]:
    phase = phase_image.astype(float)
    blur = filters.gaussian(phase, sigma=1.5, preserve_range=True)
    coarse = filters.gaussian(blur, sigma=12, preserve_range=True)
    residual_dark = coarse - blur
    edge = filters.sobel(blur)

    dark_raw = blur < filters.threshold_otsu(blur)
    residual_raw = residual_dark > filters.threshold_otsu(residual_dark)
    edge_raw = edge > filters.threshold_otsu(edge)
    edge_raw = morphology.binary_closing(edge_raw, morphology.disk(4))
    edge_raw = ndi.binary_fill_holes(edge_raw)

    return {
        "dark_otsu": center_component(dark_raw),
        "residual_dark": center_component(residual_raw),
        "edge_fill": center_component(edge_raw),
    }


def retained_frames_for_position(position_label: str) -> list[int]:
    subset = exclusion_df.loc[
        (exclusion_df["position_label"] == position_label)
        & (~exclusion_df["exclude_from_analysis"])
    ].sort_values("time_index")
    return subset["time_index"].astype(int).tolist()


def review_frames(position_label: str) -> list[int]:
    retained = retained_frames_for_position(position_label)
    if not retained:
        return []
    last_clean = retained[-1]
    selected = sorted({0, last_clean // 3, (2 * last_clean) // 3, last_clean})
    return [frame for frame in selected if frame in set(retained)]


In [ ]:
# -------------------------------
# Review frame map
# -------------------------------
review_rows = []
for position_label in REVIEW_POSITIONS:
    retained = retained_frames_for_position(position_label)
    frames = review_frames(position_label)
    review_rows.append(
        {
            "position_label": position_label,
            "retained_frame_count": len(retained),
            "last_clean_frame": retained[-1] if retained else None,
            "review_frames": frames,
        }
    )

review_df = pd.DataFrame(review_rows)
display(display_time_df(review_df[["position_label", "last_clean_frame", "review_frames"]]))


In [ ]:
# -------------------------------
# Per-position mask review figures
# -------------------------------
for position_label in REVIEW_POSITIONS:
    frames = review_frames(position_label)
    if not frames:
        continue

    fig, axes = plt.subplots(
        len(frames),
        len(MASK_METHODS),
        figsize=(4.2 * len(MASK_METHODS), 3.5 * len(frames)),
    )
    if len(frames) == 1:
        axes = np.expand_dims(axes, axis=0)
    if len(MASK_METHODS) == 1:
        axes = np.expand_dims(axes, axis=1)

    for row_index, time_index in enumerate(frames):
        phase = load_phase_frame(position_label, time_index)
        candidates = phase_mask_candidates(phase)
        for col_index, method in enumerate(MASK_METHODS):
            ax = axes[row_index, col_index]
            ax.imshow(display_image(phase), cmap="gray")
            draw_mask_contour(ax, candidates[method], color="deepskyblue", linewidth=2.2)
            ax.set_title(f"{position_label} t={format_display_hours_from_index(time_index, 0)} | {method}")
            ax.axis("off")

    fig.tight_layout()
    display(fig)
    if WRITE_OUTPUTS:
        fig.savefig(PREVIEW_DIR / f"{position_label}_retained_frame_mask_review.png", dpi=180, bbox_inches="tight")
    plt.close(fig)


In [ ]:
created_files = sorted(PREVIEW_DIR.glob("*_retained_frame_mask_review.png"))

print("Created preview files")
for path in created_files:
    print("-", path.relative_to(ROOT).as_posix())
